# 06. 손 · 얼굴 점 찍기 — MediaPipe, 그리고 점으로 규칙 만들기

> MediaPipe 는 구글의 실시간 영상 AI 묶음이다. 손마다 **점 21개**, 얼굴에 **점 478개**를 찍어 준다.
> 점 좌표만 있으면 "손가락 몇 개?", "집게?", "입 벌림?" 같은 판정을 **간단한 계산**으로 만들 수 있다.

### 이 시간에 배우는 것
1. MediaPipe Tasks 쓰는 법: 옵션 → 만들기 → `recognize` / `detect`
2. **정규화 좌표**(0~1)와 픽셀 좌표, 손 점 번호 체계
3. 점 좌표로 규칙 만들기: 손가락 세기 · 집게 · 입 벌림
4. (참고) MediaPipe 는 OpenVINO 가 아니라 자체 엔진(TFLite)으로 **CPU** 에서 돈다

### 손 점 번호
```
        8   12  16  20       끝
        7   11  15  19
        6   10  14  18
    4   5    9  13  17       뿌리
   3
  2                          0 = 손목
 1                           1~4 엄지 · 5~8 검지 · 9~12 중지 · 13~16 약지 · 17~20 새끼
0
```

## 1. 준비

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

import mediapipe as mp
from mediapipe.tasks.python import BaseOptions, vision

MP = MODELS / "mediapipe"
hands = vision.GestureRecognizer.create_from_options(vision.GestureRecognizerOptions(
    base_options=BaseOptions(model_asset_path=str(MP / "gesture_recognizer.task")), num_hands=2))
face = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=str(MP / "face_landmarker.task")), num_faces=1))

def mp_image(bgr):
    """MediaPipe 는 RGB 의 mp.Image 를 받는다."""
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))

## 2. 사진 — 손을 보여 주며 찍는다

In [ ]:
PHOTO = "sample.jpg"          # 웹캠이 없을 때 쓸 사진 (이 노트북과 같은 폴더)

def show(bgr, width=720):
    """OpenCV 사진(BGR)을 노트북 안에 보여 준다."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """웹캠에서 한 장. 웹캠이 없으면 PHOTO 파일."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # 처음 몇 장은 어둡게 나오는 웹캠이 있다
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"웹캠도 {PHOTO} 도 없어요 — PHOTO 에 사진 파일 이름을 넣으세요"
        print("웹캠이 없어 파일을 씁니다:", PHOTO)
    return frame

frame = grab()
print("사진 크기(세로, 가로, 채널):", frame.shape)
show(frame)

## 3. 손 점 21개 + 손동작
- 결과의 `hand_landmarks[i]` = i 번째 손의 점 21개. 각 점의 `x, y` 는 **0~1 비율**(사진 가로 · 세로에 대한), `z` 는 손목 기준 상대 깊이.
- `gestures[i]` = 손동작 이름과 점수. 미리 배운 7가지: `Thumb_Up`(엄지척) · `Thumb_Down` · `Victory`(브이) · `Open_Palm`(손바닥) · `Closed_Fist`(주먹) · `Pointing_Up`(검지) · `ILoveYou`.
- `handedness[i]` = 왼손/오른손. **셀카처럼 좌우가 뒤집힌 사진이면 반대로** 나온다.

In [ ]:
FINGERS = [(0, 1, 2, 3, 4), (0, 5, 6, 7, 8), (5, 9, 10, 11, 12), (9, 13, 14, 15, 16), (13, 17, 18, 19, 20), (0, 17)]

res = hands.recognize(mp_image(frame))
h, w = frame.shape[:2]
view = frame.copy()
for i, lms in enumerate(res.hand_landmarks):
    pts = [(int(p.x * w), int(p.y * h)) for p in lms]          # 0~1 → 픽셀
    for chain in FINGERS:
        for a, b in zip(chain, chain[1:]):
            cv2.line(view, pts[a], pts[b], (255, 200, 0), 2)
    for k, (x, y) in enumerate(pts):
        cv2.circle(view, (x, y), 4, (0, 0, 255), -1)
        cv2.putText(view, str(k), (x + 4, y - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 0, 0), 1)
    g = res.gestures[i][0] if res.gestures and res.gestures[i] else None
    side = res.handedness[i][0].category_name if res.handedness else "?"
    print(f"손 {i}: {side} · 손동작 {g.category_name} ({g.score:.2f})" if g else f"손 {i}: {side}")
print("찾은 손:", len(res.hand_landmarks))
show(view)

## 4. 규칙 만들기 ① — 펴진 손가락 세기
손가락 끝이 두 번째 마디보다 **위에** 있으면 편 것이다. 사진 좌표는 **아래로 갈수록 y 가 커지므로** "위" = y 가 **작다**.
엄지는 옆으로 펴지므로 x 로 비교한다 — 엄지 끝(4)이 엄지 마디(3)보다 손바닥 바깥쪽에 있으면 편 것. 바깥쪽 방향은 손목(0)과 새끼 뿌리(17)를 보고 정한다.

In [ ]:
TIPS, JOINTS = [8, 12, 16, 20], [6, 10, 14, 18]

def count_fingers(lms):
    n = sum(lms[t].y < lms[j].y for t, j in zip(TIPS, JOINTS))       # 검지~새끼: y 비교
    outward = 1 if lms[17].x < lms[0].x else -1                        # 새끼가 왼쪽에 있으면 엄지는 오른쪽으로 편다
    n += (lms[4].x - lms[3].x) * outward > 0                           # 엄지: x 비교
    return int(n)

for i, lms in enumerate(res.hand_landmarks):
    print(f"손 {i}: 펴진 손가락 {count_fingers(lms)}개")

## 5. 규칙 만들기 ② — 집게 (엄지 끝과 검지 끝 사이 거리)
거리를 픽셀로 재면 손이 카메라에서 멀어질 때 작아진다. **손 크기로 나누면**(손목~중지 뿌리 길이) 거리와 상관없는 값이 된다.

In [ ]:
def dist(a, b):
    return float(np.hypot(a.x - b.x, a.y - b.y))

for i, lms in enumerate(res.hand_landmarks):
    ratio = dist(lms[4], lms[8]) / (dist(lms[0], lms[9]) + 1e-6)     # 손 크기로 나눔
    print(f"손 {i}: 엄지-검지 비율 {ratio:.2f} → {'집게' if ratio < 0.25 else '벌림'}")

## 6. 얼굴 점 478개 — 입 벌림
얼굴 점 번호는 눈 · 눈썹 · 입술 · 윤곽마다 정해져 있다. 468~477 은 홍채(눈동자) 점이다.
13(윗입술 안쪽)과 14(아랫입술 안쪽) 사이를 얼굴 높이(10 이마 ~ 152 턱)로 나누면 입 벌림 정도가 된다.

In [ ]:
r = face.detect(mp_image(frame))
view = frame.copy()
for lms in r.face_landmarks:
    for p in lms:
        cv2.circle(view, (int(p.x * w), int(p.y * h)), 1, (0, 255, 120), -1)
    open_ratio = dist(lms[13], lms[14]) / (dist(lms[10], lms[152]) + 1e-6)
    print(f"점 {len(lms)}개 · 입 벌림 {open_ratio:.3f} → {'벌림' if open_ratio > 0.05 else '다묾'}")
show(view)

## 자주 나는 오류
| 증상 | 원인 · 고침 |
|---|---|
| `RuntimeError ... model_asset_path` | `.task` 파일이 없다 → `models/mediapipe` 확인 |
| 손을 못 찾음 | 손이 너무 작거나 잘렸다, 어둡다 → 가까이, 밝게 |
| 왼손/오른손이 반대 | 사진이 거울처럼 뒤집혀 있다 (`cv2.flip(frame, 1)` 여부) |
| 규칙이 사람마다 다르게 맞음 | 경계값(0.25, 0.05)을 실제로 찍어 보며 조정한다 |

## 정리
- MediaPipe 결과는 **0~1 비율 좌표**의 점 목록이다. 사진 크기를 곱하면 픽셀.
- 점으로 만드는 규칙은 **비율**로 만들어야 거리 · 크기에 흔들리지 않는다.
- 규칙(사람이 정한 경계값)으로 안 되면 → 07 번처럼 점 좌표를 특징으로 **학습**시킨다.

## 해 볼 것
1. 웹캠을 열어 두고 반복하며 손가락 수를 실시간으로 화면에 띄워 보자.
2. "브이(✌)" 를 손동작 이름 없이 점 규칙만으로 판정해 보자.
   <details><summary>힌트</summary>

   검지 · 중지는 펴고(끝 y < 마디 y), 약지 · 새끼는 접었는지(끝 y > 마디 y) 본다.
   </details>
3. 눈 감음을 판정해 보자 (왼눈 위 159 · 아래 145).
4. 점 21개 × (x, y, z) = 63개 숫자를 특징으로 07 번 분류기를 학습시켜 나만의 손동작을 만들어 보자.

---
# 🛠 미니 프로젝트 — 손가락 가위바위보

**만들 것**: 컴퓨터와 **가위바위보 3판**. 셋을 센 뒤 웹캠으로 내 손을 찍어 펴진 손가락 수로 판정한다.
0개 = 바위 · 2개 = 가위 · 5개 = 보 (그 밖이면 "다시").

**쓰는 것**: `hands` · `mp_image` · `count_fingers` (이 노트북), `random`

In [ ]:
import random

HAND = {0: "바위", 1: "바위", 2: "가위", 3: "가위", 4: "보", 5: "보"}     # 조금 틀려도 봐주기
WIN = {("가위", "보"), ("바위", "가위"), ("보", "바위")}                 # (이기는 것, 지는 것)
score = {"나": 0, "컴퓨터": 0}

for rnd in range(1, 4):
    for c in ("가위", "바위", "보!"):
        print(c, end=" ", flush=True); time.sleep(0.7)
    shot = grab()
    r = hands.recognize(mp_image(shot))
    if not r.hand_landmarks:
        print(f"\n{rnd}판: 손이 안 보여요 — 다시"); continue
    n = count_fingers(r.hand_landmarks[0])
    me, com = HAND.get(n), random.choice(["가위", "바위", "보"])
    if (me, com) in WIN:
        score["나"] += 1; res = "이겼다!"
    elif (com, me) in WIN:
        score["컴퓨터"] += 1; res = "졌다"
    else:
        res = "비김"
    print(f"\n{rnd}판: 나 {me}(손가락 {n}) vs 컴퓨터 {com} → {res}")
    show(shot, 240)
print("최종:", score)

**더 해 보기**
- 손동작 이름(`gestures`)으로 판정해 보고, 손가락 수 방식과 어느 쪽이 더 잘 맞는지 비교
- 컴퓨터가 내 지난 손을 기억해 이기는 쪽을 더 자주 내게 만들기
- 06 의 손가락 수 대신 07 의 나만의 분류기로 판정하기